In [3]:
import pandas as pd
import numpy as np
import sys, os
from helpers import Condition, Trial
from helpers.metadata import AVOIDING_THRESHOLD,PIXEL_RAILWAY_WIDTH

In [8]:
FILE_PATH = "preprocessed"

# Preprocess singleconditions

In [21]:
conditions = Condition.load_conditional_groups()

In [22]:
### try to include boundaries for left_handed_people

def transform_to_normalized_distances(trial: Trial) -> Trial:
    """Nomalize to start = 0 and target = 1"""

    tx, ty = trial.get_target_in_cm()
    sx, _ = trial.get_start()
    trajectory_px = trial.get_trajectory_data()[["current_pos_x"]]

    trial[["current_pos_x_cm", "current_pos_y_cm"]] = trial.get_trajectory_data_in_cm()[["current_pos_x", "current_pos_y"]]
    trial[["current_pos_x_normalized", "current_pos_y_normalized"]] = trial.get_trajectory_data_normalized()[["current_pos_x", "current_pos_y"]]

    trial[["target_pos_x_cm"]] = tx
    trial[["target_pos_y_cm"]] = ty


    if sx <= 0:
        trial["trial_rail_pos"] = (trajectory_px + np.abs(sx)) / PIXEL_RAILWAY_WIDTH
    else :
        trial["trial_rail_pos"] = ((-(trajectory_px)) + sx) / PIXEL_RAILWAY_WIDTH

    return trial

In [23]:
threshold = AVOIDING_THRESHOLD * PIXEL_RAILWAY_WIDTH
def add_trajectory_side(trial:
     Trial) -> Trial:
    """appends trial dataframe by column to store whether side was switched"""

    trial["switched_side"] = 0 
    trial.loc[trial["trial_rail_pos"] >= AVOIDING_THRESHOLD, "switched_side"] = 1
    return trial

In [24]:
def drop_unneccessary(trial: Trial):
    trial = trial.drop(["left_button_pressed","middle_button_pressed","right_button_pressed", "current_pos_x", "current_pos_y", "target_pos_x", "target_pos_y"], axis=1)
    return trial

In [25]:
def get_relevant_row(trial: Trial) -> pd.DataFrame:
    """Gets the actual row per trial"""

    trial = transform_to_normalized_distances(trial)
    trial = add_trajectory_side(trial)
    type = trial.loc[:, "task"].iloc[0]
    
    if type == "avoiding":
        switched = trial.loc[trial["switched_side"] == 1]
        return switched.iloc[0] if len(switched) > 0 else trial.iloc[-1]
    elif type == "reaching":
        return trial.iloc[-1]
    else:
        raise ValueError(f"Faulty trial type found! No typ of {type}")


def find_natural_mapping(trials: list[Trial]):
    """Finding natural mapping by identifiying correlation between target and actual. negative means higher target leads to lower estimate"""
    
    relevant_rows = list(map(get_relevant_row, trials))
    pos = np.array(list(map(lambda trial: np.array([trial[["target_pos_y"]].iloc[-1], trial[["current_pos_y"]].iloc[-1]]),  relevant_rows)))
    
    X = pos[:, 0]
    y = pos[:, 1]
    corr = np.corrcoef(X, y)[0, 1] # returns covariance matrix
    
    task_mapping = trials[0].get_mapping()

    THRESHOLD = .33
    if (task_mapping == "reversed" and corr >= THRESHOLD) or (task_mapping == "direct" and corr <= THRESHOLD):
        return "reversed"
    elif (task_mapping == "direct" and corr >= THRESHOLD) or (task_mapping == "reversed" and corr <= THRESHOLD):
        return "direct"
    else:
        return "random"
    

In [ ]:
RAIL_WIDTH = Trial._transform_px_to_cm(PIXEL_RAILWAY_WIDTH, 0)

def is_outlier(trial: Trial):
    task = trial["task"].iloc[0]
    row = get_relevant_row(trial)

    #Any scan out of bounds
    if trial["current_pos_x_cm"][(trial["current_pos_x_cm"] < -0.25) | (trial["current_pos_x_cm"] > RAIL_WIDTH + 0.25)].count() > 0:
        return True

    if task == "reaching":
        return row["switched_side"] == 1
    
    elif task == "avoiding":
        return trial["switched_side"].sum() == 0
    
    else:
        raise ValueError(f"Invalid task: {task}")
    
    return False

2.3325


In [27]:
def process(trial:Trial, natural_mapping: str) -> Trial:
    trial = transform_to_normalized_distances(trial)
    trial = add_trajectory_side(trial)
    trial["natural_mapping"] = natural_mapping

    if is_outlier(trial):
        return None
    
    trial = drop_unneccessary(trial)

    return trial


In [ ]:
OUTLIERS = 0

if not os.path.exists(FILE_PATH):
    os.mkdir(FILE_PATH)

participants = 0
for condition in conditions:
    participants += condition.get_participant_count()
 
participant_count = 0
for condition in conditions:

    p_processed = []
    for participant in condition:
        participant_count += 1
        sys.stdout.write(f"\rProcessing {participant.get_participant_id()} [{participant_count}|{participants}]\t")

        pre_test = participant.get_as_one_list(phase="Pre-Test", block=1)
        natural_mapping = find_natural_mapping(pre_test)

        for entry in list(map(lambda trial: process(trial, natural_mapping), participant)):
            if entry is None:
                OUTLIERS += 1
                continue

            p_processed.append(entry)

    sys.stdout.write(f"\rConcatinating {condition.get_group_name()}\t")
    df_joint = pd.concat(p_processed).round(3)
    df_joint.to_csv(f"{FILE_PATH}/{condition.get_group_name()}.csv")

sys.stdout.write(f"\nDone\n")
sys.stdout.write(f"\n{OUTLIERS} outliers detected and removed\n")
sys.stdout.flush()

Concatinating ai	49|49]	
Done

773 outliers detected and removed


In [29]:
del conditions

# Detect relevant rows

In [4]:
class PreprocessedCondition(pd.DataFrame):

    def __init__(self, file: str, condition:str):
        super().__init__(pd.read_csv(file))
        self.condition = condition

    def get_condition(self):
        return self.condition

In [31]:
rd = PreprocessedCondition(f"{FILE_PATH}/rd.csv", "rd")
ri = PreprocessedCondition(f"{FILE_PATH}/ri.csv", "ri")
ad = PreprocessedCondition(f"{FILE_PATH}/ad.csv", "ad")
ai = PreprocessedCondition(f"{FILE_PATH}/ai.csv", "ai")

In [32]:
def get_relevant_row(df: pd.DataFrame):
    """Gets the actual row per trial"""
    type = df.loc[:, "task"].iloc[0]
    if type == "avoiding":
        switched = df.loc[df["switched_side"] == 1]
        return switched.iloc[0] if len(switched) > 0 else df.iloc[-1]
    elif type == "reaching":
        return df.iloc[-1]
    else:
        raise ValueError(f"Faulty trial type found! No typ of {type}")

def find_relevant_entry(df: pd.DataFrame):
    """Separates the phase df into blocks and returns one row per block"""
    #print(df)
    return df.groupby(["trial_index"]).apply(get_relevant_row, include_groups=False)

In [33]:
for cond in [ai, ad, rd, ri]:
    nested_group = cond.groupby(["participant_id", "phase", "block"])
    grouped = nested_group.apply(find_relevant_entry, include_groups=False)
    grouped.to_csv(f"{FILE_PATH}/{cond.get_condition()}_relevant_trials.csv")

In [34]:
del ai, ad, rd, ri

# Calculate Block Data

In [9]:
ri = PreprocessedCondition(f"{FILE_PATH}/ri_relevant_trials.csv", "ri")
rd = PreprocessedCondition(f"{FILE_PATH}/rd_relevant_trials.csv", "rd")
ai = PreprocessedCondition(f"{FILE_PATH}/ai_relevant_trials.csv", "ai")
ad = PreprocessedCondition(f"{FILE_PATH}/ad_relevant_trials.csv", "ad")

In [10]:
def get_mean_variance_median_per_block(df: pd.DataFrame):
    df["difference"] = np.abs(df.loc[:,"target_pos_y_cm"] - df.loc[:,"current_pos_y_cm"])
    return pd.DataFrame({
        "diff_mean": [df["difference"].mean()],
        "diff_variance": [df["difference"].var()],
        "mapping": [df["mapping"].iloc[0]],
        "natural_mapping": [df["natural_mapping"].iloc[0]]
    })

In [11]:
for cond in [ai, ad, ri, rd]:
    grouped = cond.groupby(["participant_id", "phase", "block", "task"]).apply(get_mean_variance_median_per_block, include_groups=False)
    grouped.to_csv(f"{FILE_PATH}/{cond.get_condition()}_block_measures.csv")